# 🏛️ HEIST ARCHITECT v2 — Adversarial RL on Dual T4

**Fixes & upgrades over v1:**
- ✅ Real PPO training loop (not just softmax outputs)
- ✅ Architect on GPU:0, Robber on GPU:1 (true split, not DataParallel)
- ✅ Self-play curriculum: agents fight progressively harder past versions of themselves
- ✅ ELO rating system tracking who's actually winning
- ✅ LSTM hidden state managed correctly across rollouts
- ✅ Reward shaping + GAE advantage estimation
- ✅ HuggingFace auto-checkpoint with metrics + model weights
- ✅ Live training dashboard (loss curves, ELO, win rates)


## 📦 1. Install & Imports

In [1]:
!pip install -q huggingface_hub gymnasium matplotlib rich

# Clone your repo so we can reuse the environment
import os
if not os.path.exists('heist_repo'):
    os.system('git clone https://github.com/Shanmuk4622/RL-Project-Heist-Architect-Adversarial-Reinforcement-Learning-Framework-CSE4019.git heist_repo')
import sys
sys.path.insert(0, '/kaggle/working/heist_repo')
print('✅ Repo ready')

Cloning into 'heist_repo'...


✅ Repo ready


In [2]:
import math, time, json, copy, random
from collections import deque, OrderedDict
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print('✅ All imports OK')

✅ All imports OK


## 🖥️ 2. Dual GPU Setup (True Split — Not DataParallel)

In [3]:
# ── The v1 notebook used nn.DataParallel which just mirrors the same model.
# ── We instead assign each AGENT to its own GPU so they train independently.
# ── This lets each agent use ~12GB VRAM instead of sharing 12GB.

n_gpu = torch.cuda.device_count()
print(f'GPUs available: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

if n_gpu >= 2:
    DEV_ARCH   = torch.device('cuda:0')   # Architect lives here
    DEV_ROBBER = torch.device('cuda:1')   # Robber lives here
    print('✅ Dual GPU: Architect→cuda:0  |  Robber→cuda:1')
elif n_gpu == 1:
    DEV_ARCH = DEV_ROBBER = torch.device('cuda:0')
    print('⚠️  Single GPU — both agents share cuda:0')
else:
    DEV_ARCH = DEV_ROBBER = torch.device('cpu')
    print('❌ CPU only — training will be slow')

GPUs available: 2
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB
✅ Dual GPU: Architect→cuda:0  |  Robber→cuda:1


## 🤗 3. HuggingFace Setup

In [ ]:
from huggingface_hub import HfApi, login

HF_TOKEN  = 'REPLACE_WITH_HF_TOKEN'
REPO_ID   = 'Shanmuk4622/heist-architect-v2'
CKPT_DIR  = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

try:
    login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi()
    api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)
    HF_AVAILABLE = True
    print(f'✅ HF repo ready: https://huggingface.co/{REPO_ID}')
except Exception as e:
    HF_AVAILABLE = False
    api = None
    print(f'⚠️  HuggingFace unavailable: {e}')
    print('   → Enable Internet in Kaggle: Settings → Internet → On')
    print('   → Training will still run; checkpoints saved locally only.')

def save_to_hf(arch_net, robber_net, metrics: dict, episode: int):
    tag = f'ep{episode:05d}'
    d = f'{CKPT_DIR}/{tag}'
    os.makedirs(d, exist_ok=True)

    # Strip DataParallel wrapper if present
    def unwrap(m):
        return m.module if hasattr(m, 'module') else m

    torch.save(unwrap(arch_net).state_dict(),   f'{d}/architect.pt')
    torch.save(unwrap(robber_net).state_dict(), f'{d}/robber.pt')

    with open(f'{d}/metrics.json', 'w') as f:
        json.dump({'episode': episode, **metrics}, f, indent=2)

    if not HF_AVAILABLE:
        print(f'  💾 Local checkpoint saved: {d}  (HF offline)')
        return
    try:
        api.upload_folder(
            folder_path=d, repo_id=REPO_ID, path_in_repo=tag,
            token=HF_TOKEN,
            commit_message=f'ep{episode} | robber_win={metrics.get("robber_win_rate",0):.2f} | elo_diff={metrics.get("elo_diff",0):.0f}'
        )
        print(f'  ✅ HF checkpoint pushed: {tag}')
    except Exception as e:
        print(f'  ⚠️  HF push failed: {e}')

✅ HF repo ready: https://huggingface.co/Shanmuk4622/heist-architect-v2


In [ ]:
# from huggingface_hub import HfApi, login

# HF_TOKEN = "REPLACE_WITH_HF_TOKEN"   # replace with your real token
# REPO_ID = "Shanmuk4622/heist-architect-v2"

# login(token=HF_TOKEN)
# api = HfApi()

# # List all files in the repo
# files = api.list_repo_files(repo_id=REPO_ID)
# print("Current files:", files)

# # Delete all folders starting with 'ep' (checkpoints)
# for f in files:
#     if f.startswith("ep"):
#         api.delete_file(path_in_repo=f, repo_id=REPO_ID, token=HF_TOKEN)
#         print(f"Deleted: {f}")

# print("✅ All checkpoints removed from HF repo.")

Current files: ['.gitattributes', 'ep00500/architect.pt', 'ep00500/metrics.json', 'ep00500/robber.pt', 'ep01000/architect.pt', 'ep01000/metrics.json', 'ep01000/robber.pt', 'ep01500/architect.pt', 'ep01500/metrics.json', 'ep01500/robber.pt', 'ep02000/architect.pt', 'ep02000/metrics.json', 'ep02000/robber.pt', 'ep02500/architect.pt', 'ep02500/metrics.json', 'ep02500/robber.pt']
Deleted: ep00500/architect.pt
Deleted: ep00500/metrics.json
Deleted: ep00500/robber.pt
Deleted: ep01000/architect.pt
Deleted: ep01000/metrics.json
Deleted: ep01000/robber.pt
Deleted: ep01500/architect.pt
Deleted: ep01500/metrics.json
Deleted: ep01500/robber.pt
Deleted: ep02000/architect.pt
Deleted: ep02000/metrics.json
Deleted: ep02000/robber.pt
Deleted: ep02500/architect.pt
Deleted: ep02500/metrics.json
Deleted: ep02500/robber.pt
✅ All checkpoints removed from HF repo.


## ⚙️ 4. Config

In [41]:
@dataclass
class Config:
    # Environment
    grid_rows: int = 20
    grid_cols: int = 20
    n_cameras: int = 6
    n_guards:  int = 3
    max_steps: int = 300

    # Networks
    cnn_channels: List[int] = field(default_factory=lambda: [64, 128, 256, 256])
    lstm_hidden:  int = 512
    input_channels: int = 12   # matches your existing env channel count

    # PPO
    lr_arch:      float = 1e-4
    lr_robber:    float = 3e-4
    gamma:        float = 0.99
    gae_lambda:   float = 0.95
    clip_eps:     float = 0.2
    entropy_coef: float = 0.02
    value_coef:   float = 0.5
    max_grad_norm:float = 0.5
    ppo_epochs:   int   = 8
    rollout_len:  int   = 512    # steps per rollout before update
    minibatch_sz: int   = 128
    lr_robber:    float = 5e-4      # was 3e-4
    entropy_coef: float = 0.05      # was 0.02

    # Self-play
    selfplay_pool_size: int   = 10    # keep last N snapshots
    selfplay_use_latest: float = 0.7  # 70% of time play vs latest, 30% vs random past

    # Curriculum (robber win-rate threshold to advance stage)
    curriculum_stages: List[dict] = field(default_factory=lambda: [
        {'name': 'Rookie',      'n_cameras': 2, 'n_guards': 1},
        {'name': 'Intermediate','n_cameras': 4, 'n_guards': 2},
        {'name': 'Expert',      'n_cameras': 6, 'n_guards': 3},
        {'name': 'Master',      'n_cameras': 8, 'n_guards': 4},
    ])
    curriculum_win_thresh: float = 0.55

    # Training schedule
    total_episodes:    int = 20000
    checkpoint_every:  int = 500
    log_every:         int = 50
    eval_every:        int = 200
    eval_episodes:     int = 20

CFG = Config()
print('Config ready. Curriculum stages:')
for s in CFG.curriculum_stages:
    print(f"  {s['name']}: cameras={s['n_cameras']} guards={s['n_guards']}")

Config ready. Curriculum stages:
  Rookie: cameras=2 guards=1
  Intermediate: cameras=4 guards=2
  Expert: cameras=6 guards=3
  Master: cameras=8 guards=4


## 🏗️ 5. Environment Wrapper

In [42]:
# ── Environment Wrapper (Fixed) ──────────────────────────────────────────────
import numpy as np

try:
    from heist_repo.heist_architect.environment import HeistEnvironment, EnvironmentConfig
    USING_ORIGINAL_ENV = True
    print('✅ Using original HeistEnvironment from repo')
except Exception as e:
    USING_ORIGINAL_ENV = False
    print(f'⚠️  Could not import original env ({e}) — using standalone fallback env')

# -----------------------------------------------------------------------------
# Standalone fallback environment (only used if original import fails)
if not USING_ORIGINAL_ENV:
    class HeistEnvStandalone:
        """
        Standalone grid environment for when the repo import fails.
        Obs: (input_channels, grid_rows, grid_cols) float32 tensor.
        Robber actions: 0=up 1=down 2=left 3=right
        Architect actions: (camera_positions, guard_positions) flattened index
        """
        EMPTY=0; WALL=1; VAULT=2; CAMERA=3; GUARD=4; ROBBER=5

        def __init__(self, cfg):
            self.cfg = cfg
            self.R, self.C = cfg.grid_rows, cfg.grid_cols
            self.reset()

        def reset(self, n_cameras=None, n_guards=None):
            nc = n_cameras or self.cfg.n_cameras
            ng = n_guards  or self.cfg.n_guards
            R, C = self.R, self.C
            self.grid = np.zeros((R, C), dtype=np.int32)
            self.steps = 0
            self.done  = False

            rng = np.random.default_rng()
            # Walls
            mask = rng.random((R,C)) < 0.10
            mask[0,:]=mask[-1,:]=mask[:,0]=mask[:,-1]=False
            self.grid[mask] = self.WALL

            def place(obj, n):
                positions=[]
                for _ in range(n):
                    for _ in range(500):
                        r,c=rng.integers(1,R-1),rng.integers(1,C-1)
                        if self.grid[r,c]==self.EMPTY:
                            self.grid[r,c]=obj
                            positions.append([r,c])
                            break
                return positions

            self.vault_positions  = place(self.VAULT,  2)
            self.camera_positions = place(self.CAMERA, nc)
            self.guard_positions  = place(self.GUARD,  ng)

            # Robber spawns at top-left area
            self.robber_pos = [1, 1]
            if self.grid[1,1] == self.EMPTY:
                self.grid[1,1] = self.ROBBER

            self.vaults_robbed = 0
            return self._obs()

        def _camera_coverage(self):
            covered = set()
            for (cr,cc) in self.camera_positions:
                for dr in range(-2,3):
                    for dc in range(-2,3):
                        r,c=cr+dr,cc+dc
                        if 0<=r<self.R and 0<=c<self.C:
                            covered.add((r,c))
            return covered

        def _obs(self):
            # Build (12, R, C) tensor: one channel per feature
            obs = np.zeros((12, self.R, self.C), dtype=np.float32)
            obs[0] = (self.grid == self.EMPTY).astype(np.float32)
            obs[1] = (self.grid == self.WALL).astype(np.float32)
            obs[2] = (self.grid == self.VAULT).astype(np.float32)
            obs[3] = (self.grid == self.CAMERA).astype(np.float32)
            obs[4] = (self.grid == self.GUARD).astype(np.float32)
            obs[5] = (self.grid == self.ROBBER).astype(np.float32)
            # Camera coverage map
            cov = self._camera_coverage()
            for (r,c) in cov:
                obs[6,r,c] = 1.0
            # Robber distance map (normalized)
            rr,rc = self.robber_pos
            for r in range(self.R):
                for c in range(self.C):
                    obs[7,r,c] = 1.0 - (abs(r-rr)+abs(c-rc))/(self.R+self.C)
            # Vault proximity
            for (vr,vc) in self.vault_positions:
                obs[8,vr,vc] = 1.0
            # Step progress
            obs[9] = self.steps / self.cfg.max_steps
            # Vaults robbed ratio
            obs[10] = self.vaults_robbed / max(len(self.vault_positions)+self.vaults_robbed, 1)
            # Guard proximity
            for (gr,gc) in self.guard_positions:
                obs[11,gr,gc] = 1.0
            return obs

        def step_robber(self, action: int):
            if self.done: return self._obs(), 0.0, True, {}
            self.steps += 1
            R,C = self.R, self.C
            dr,dc = [(-1,0),(1,0),(0,-1),(0,1)][action]
            nr = np.clip(self.robber_pos[0]+dr, 0, R-1)
            nc = np.clip(self.robber_pos[1]+dc, 0, C-1)

            reward = -0.01  # step cost
            info = {'caught':False,'robbed':False,'alarm':False}

            if self.grid[nr,nc] != self.WALL:
                if self.grid[self.robber_pos[0], self.robber_pos[1]] == self.ROBBER:
                    self.grid[self.robber_pos[0], self.robber_pos[1]] = self.EMPTY
                self.robber_pos = [nr, nc]
                self.grid[nr,nc] = self.ROBBER

            # Guard catch
            for gp in self.guard_positions:
                if gp[0]==self.robber_pos[0] and gp[1]==self.robber_pos[1]:
                    reward = -10.0; self.done=True; info['caught']=True
                    return self._obs(), reward, True, info

            # Camera detection
            if tuple(self.robber_pos) in self._camera_coverage():
                reward -= 1.0; info['alarm']=True

            # Vault
            for i,(vr,vc) in enumerate(self.vault_positions):
                if vr==self.robber_pos[0] and vc==self.robber_pos[1]:
                    self.vaults_robbed += 1
                    self.vault_positions.pop(i)
                    self.grid[vr,vc] = self.EMPTY
                    reward += 10.0; info['robbed']=True
                    if not self.vault_positions:
                        reward += 20.0; self.done=True
                    break

            if self.steps >= self.cfg.max_steps:
                self.done = True

            return self._obs(), reward, self.done, info

        def step_architect(self, cam_actions, guard_actions):
            """Move cameras and guards. Actions are flat grid indices."""
            R,C = self.R, self.C
            dirs = [(-1,0),(1,0),(0,-1),(0,1),(0,0)]

            # Reposition cameras ONLY on the first step (layout phase)
            if getattr(self, 'steps', 0) == 0:
                for i, pos_idx in enumerate(cam_actions):
                    r, c = pos_idx // C, pos_idx % C
                    if i < len(self.camera_positions):
                        old = self.camera_positions[i]
                        if self.grid[old[0],old[1]] == self.CAMERA:
                            self.grid[old[0],old[1]] = self.EMPTY
                        if self.grid[r,c] == self.EMPTY:
                            self.camera_positions[i] = [r,c]
                            self.grid[r,c] = self.CAMERA

            # Move guards
            for i, a in enumerate(guard_actions):
                if i >= len(self.guard_positions): break
                dr,dc = dirs[a % len(dirs)]
                gr,gc = self.guard_positions[i]
                nr = np.clip(gr+dr, 0, R-1)
                nc_g = np.clip(gc+dc, 0, C-1)
                if self.grid[nr,nc_g] not in [self.WALL,self.CAMERA,self.GUARD]:
                    if self.grid[gr,gc] == self.GUARD:
                        self.grid[gr,gc] = self.EMPTY
                    self.guard_positions[i] = [nr,nc_g]
                    self.grid[nr,nc_g] = self.GUARD

    def make_env(cfg: Config, n_cameras=None, n_guards=None):
        e = HeistEnvStandalone(cfg)
        e.reset(n_cameras=n_cameras, n_guards=n_guards)
        return e

else:
    def make_env(cfg: Config, n_cameras=None, n_guards=None):
        env_cfg = EnvironmentConfig(grid_rows=cfg.grid_rows, grid_cols=cfg.grid_cols)
        # If your EnvironmentConfig supports setting cameras/guards, add here
        return HeistEnvironment(env_cfg)

# -----------------------------------------------------------------------------
# Observation processing (handles both original dict and standalone array)
def process_observation(obs, env=None):
    """Convert any observation to a (12, H, W) float32 numpy array."""
    if not USING_ORIGINAL_ENV:
        # Standalone env already returns the correct shape
        return np.array(obs, dtype=np.float32)

    # --- Handle numpy array from original environment ---
    if isinstance(obs, np.ndarray):
        # If already a 3D array with 12 channels, use as is
        if obs.ndim == 3 and obs.shape[0] == 12:
            return obs.astype(np.float32)
        # If a 2D grid (H, W), convert to 12-channel representation
        elif obs.ndim == 2:
            grid = obs.astype(np.int32)
            H, W = grid.shape
            channels = np.zeros((12, H, W), dtype=np.float32)

            # Object type encoding (adjust values if your env uses different IDs)
            EMPTY, WALL, VAULT, CAMERA, GUARD, ROBBER = 0, 1, 2, 3, 4, 5
            channels[0] = (grid == EMPTY).astype(np.float32)
            channels[1] = (grid == WALL).astype(np.float32)
            channels[2] = (grid == VAULT).astype(np.float32)
            channels[3] = (grid == CAMERA).astype(np.float32)
            channels[4] = (grid == GUARD).astype(np.float32)
            channels[5] = (grid == ROBBER).astype(np.float32)

            # Camera coverage: approximate if positions not available
            coverage = np.zeros((H, W), dtype=np.float32)
            cam_pos = np.argwhere(grid == CAMERA)
            for r, c in cam_pos:
                r0, r1 = max(0, r-2), min(H, r+3)
                c0, c1 = max(0, c-2), min(W, c+3)
                coverage[r0:r1, c0:c1] = 1.0
            channels[6] = coverage

            # Robber distance
            robber_pos = np.argwhere(grid == ROBBER)
            rr, rc = robber_pos[0] if len(robber_pos) > 0 else (1, 1)
            for r in range(H):
                for c in range(W):
                    channels[7, r, c] = 1.0 - (abs(r - rr) + abs(c - rc)) / (H + W)

            # Vault positions
            vaults = np.argwhere(grid == VAULT)
            for vr, vc in vaults:
                channels[8, vr, vc] = 1.0

            # Step progress (unknown from static grid; set to 0)
            channels[9] = 0.0

            # Vaults robbed ratio (unknown)
            channels[10] = 0.0

            # Guard proximity
            guards = np.argwhere(grid == GUARD)
            for gr, gc in guards:
                channels[11, gr, gc] = 1.0

            return channels
        else:
            raise ValueError(f"Unexpected numpy array shape: {obs.shape}")

    # Original environment returns a dictionary
    if isinstance(obs, dict):
        grid = np.array(obs.get('grid', obs.get('map', [[0]])), dtype=np.int32)
        H, W = grid.shape
        channels = np.zeros((12, H, W), dtype=np.float32)

        # Object type encoding
        EMPTY, WALL, VAULT, CAMERA, GUARD, ROBBER = 0, 1, 2, 3, 4, 5
        channels[0] = (grid == EMPTY).astype(np.float32)
        channels[1] = (grid == WALL).astype(np.float32)
        channels[2] = (grid == VAULT).astype(np.float32)
        channels[3] = (grid == CAMERA).astype(np.float32)
        channels[4] = (grid == GUARD).astype(np.float32)
        channels[5] = (grid == ROBBER).astype(np.float32)

        # Camera coverage
        if 'camera_coverage' in obs:
            coverage = np.array(obs['camera_coverage'], dtype=np.float32)
        else:
            coverage = np.zeros((H, W), dtype=np.float32)
            if 'camera_positions' in obs:
                for r, c in obs['camera_positions']:
                    r0, r1 = max(0, r-2), min(H, r+3)
                    c0, c1 = max(0, c-2), min(W, c+3)
                    coverage[r0:r1, c0:c1] = 1.0
        channels[6] = coverage

        # Robber distance
        if 'robber_pos' in obs:
            rr, rc = obs['robber_pos']
        else:
            robber_pos = np.argwhere(grid == ROBBER)
            rr, rc = robber_pos[0] if len(robber_pos) > 0 else (1, 1)
        for r in range(H):
            for c in range(W):
                channels[7, r, c] = 1.0 - (abs(r - rr) + abs(c - rc)) / (H + W)

        # Vault positions
        if 'vault_positions' in obs:
            vaults = obs['vault_positions']
        else:
            vaults = np.argwhere(grid == VAULT)
        for vr, vc in vaults:
            channels[8, vr, vc] = 1.0

        # Step progress
        steps = obs.get('steps', obs.get('step', 0))
        channels[9] = steps / CFG.max_steps

        # Vaults robbed ratio (if tracked)
        channels[10] = 0.0

        # Guard proximity
        if 'guard_positions' in obs:
            guards = obs['guard_positions']
        else:
            guards = np.argwhere(grid == GUARD)
        for gr, gc in guards:
            channels[11, gr, gc] = 1.0

        return channels

    # Fallback for objects with attributes
    if hasattr(obs, 'grid'):
        grid = np.array(obs.grid, dtype=np.int32)
        fake_dict = {'grid': grid}
        if hasattr(obs, 'camera_positions'): fake_dict['camera_positions'] = obs.camera_positions
        if hasattr(obs, 'guard_positions'): fake_dict['guard_positions'] = obs.guard_positions
        if hasattr(obs, 'robber_pos'): fake_dict['robber_pos'] = obs.robber_pos
        if hasattr(obs, 'vault_positions'): fake_dict['vault_positions'] = obs.vault_positions
        if hasattr(obs, 'steps'): fake_dict['steps'] = obs.steps
        return process_observation(fake_dict, env)

    raise TypeError(f"Unsupported observation type: {type(obs)}")

# -----------------------------------------------------------------------------
# Test the conversion
test_env = make_env(CFG)
raw_obs = test_env.reset()
obs_processed = process_observation(raw_obs)
print(f'✅ Processed obs shape: {obs_processed.shape}')  # Expect (12, 20, 20)

✅ Using original HeistEnvironment from repo
✅ Processed obs shape: (12, 1, 1)


## 🧠 6. Neural Network Architectures

**Why this is better than v1:**
- v1 used `nn.MaxPool2d` which loses spatial precision — bad for strategy games
- We use strided convolutions + skip connections instead
- The LSTM hidden state is properly carried across timesteps (v1 reset it every step)
- Actor and critic share a trunk — more parameter-efficient


In [43]:
# ── Shared building block ───────────────────────────────────────────────────
class ResBlock(nn.Module):
    """Residual block with group norm (works with any batch size, including 1)."""
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(min(8, ch), ch),
            nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, ch), ch),
            nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
        )
    def forward(self, x):
        return x + self.net(x)


class CNNEncoder(nn.Module):
    """Encodes (C, H, W) grid observation into a flat feature vector."""
    def __init__(self, in_ch=12, channels=(64,128,256), out_dim=512):
        super().__init__()
        layers = []
        prev = in_ch
        for ch in channels:
            layers += [
                nn.Conv2d(prev, ch, 3, stride=2, padding=1, bias=False),
                ResBlock(ch),
                ResBlock(ch),
            ]
            prev = ch
        layers.append(nn.AdaptiveAvgPool2d((3, 3)))
        self.net = nn.Sequential(*layers)
        self.proj = nn.Linear(prev * 9, out_dim)

    def forward(self, x):  # x: (B, C, H, W)
        h = self.net(x).flatten(1)
        return F.silu(self.proj(h))  # (B, out_dim)


# ── Robber: LSTM Actor-Critic ───────────────────────────────────────────────
# Uses LSTM so it can remember guard patrol patterns over time.
class RobberNet(nn.Module):
    def __init__(self, in_ch=12, cnn_out=512, lstm_hidden=512, n_actions=4):
        super().__init__()
        self.lstm_hidden = lstm_hidden
        self.encoder = CNNEncoder(in_ch, channels=(64,128,256), out_dim=cnn_out)
        self.lstm    = nn.LSTMCell(cnn_out, lstm_hidden)
        self.actor   = nn.Sequential(
            nn.Linear(lstm_hidden, 256), nn.SiLU(),
            nn.Linear(256, n_actions)
        )
        self.critic  = nn.Sequential(
            nn.Linear(lstm_hidden, 256), nn.SiLU(),
            nn.Linear(256, 1)
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        # small init for output layers
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)

    def initial_state(self, batch=1, device=None):
        dev = device or next(self.parameters()).device
        return (torch.zeros(batch, self.lstm_hidden, device=dev),
                torch.zeros(batch, self.lstm_hidden, device=dev))

    def forward(self, obs, hx, cx):
        """obs: (B,C,H,W)  hx/cx: (B, lstm_hidden)"""
        feat = self.encoder(obs)
        hx, cx = self.lstm(feat, (hx, cx))
        logits = self.actor(hx)
        value  = self.critic(hx).squeeze(-1)
        return logits, value, hx, cx

    def act(self, obs, hx, cx, greedy=False):
        with torch.no_grad():
            logits, value, hx, cx = self.forward(obs, hx, cx)
        dist = Categorical(logits=logits)
        action = logits.argmax(-1) if greedy else dist.sample()
        return action.item(), dist.log_prob(action), value, hx, cx


# ── Architect: layout + patrol policy ──────────────────────────────────────
# The architect decides WHERE to put cameras and HOW to move guards.
# We give it two separate heads so it can learn each task independently.
class ArchitectNet(nn.Module):
    def __init__(self, in_ch=12, cnn_out=512, lstm_hidden=512,
                 n_cameras=6, n_guards=3, grid_size=400):
        super().__init__()
        self.lstm_hidden = lstm_hidden
        self.n_cameras = n_cameras
        self.n_guards  = n_guards
        self.grid_size = grid_size  # R*C flat

        self.encoder = CNNEncoder(in_ch, channels=(64,128,256), out_dim=cnn_out)
        self.lstm    = nn.LSTMCell(cnn_out, lstm_hidden)

        # Camera placement head: one distribution over all grid cells per camera
        # We share weights and output n_cameras logit vectors
        self.cam_head   = nn.Sequential(
            nn.Linear(lstm_hidden, 512), nn.SiLU(),
            nn.Linear(512, n_cameras * grid_size)
        )
        # Guard movement head: 5 directions per guard
        self.guard_head = nn.Sequential(
            nn.Linear(lstm_hidden, 256), nn.SiLU(),
            nn.Linear(256, n_guards * 5)
        )
        # Critic
        self.critic = nn.Sequential(
            nn.Linear(lstm_hidden, 256), nn.SiLU(),
            nn.Linear(256, 1)
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.cam_head[-1].weight, 0.01)
        nn.init.orthogonal_(self.guard_head[-1].weight, 0.01)
        nn.init.orthogonal_(self.critic[-1].weight, 1.0)

    def initial_state(self, batch=1, device=None):
        dev = device or next(self.parameters()).device
        return (torch.zeros(batch, self.lstm_hidden, device=dev),
                torch.zeros(batch, self.lstm_hidden, device=dev))

    def forward(self, obs, hx, cx):
        feat = self.encoder(obs)
        hx, cx = self.lstm(feat, (hx, cx))

        cam_logits   = self.cam_head(hx).view(-1, self.n_cameras, self.grid_size)
        guard_logits = self.guard_head(hx).view(-1, self.n_guards, 5)
        value        = self.critic(hx).squeeze(-1)
        return cam_logits, guard_logits, value, hx, cx

    def act(self, obs, hx, cx, greedy=False):
        with torch.no_grad():
            cam_logits, guard_logits, value, hx, cx = self.forward(obs, hx, cx)
        cam_dists   = [Categorical(logits=cam_logits[0, i]) for i in range(self.n_cameras)]
        guard_dists = [Categorical(logits=guard_logits[0, i]) for i in range(self.n_guards)]
        if greedy:
            cam_actions   = [d.probs.argmax().item() for d in cam_dists]
            guard_actions = [d.probs.argmax().item() for d in guard_dists]
        else:
            cam_actions   = [d.sample().item() for d in cam_dists]
            guard_actions = [d.sample().item() for d in guard_dists]
        log_prob = sum(cam_dists[i].log_prob(torch.tensor(cam_actions[i], device=obs.device))
                       for i in range(self.n_cameras))
        log_prob += sum(guard_dists[i].log_prob(torch.tensor(guard_actions[i], device=obs.device))
                        for i in range(self.n_guards))
        entropy = sum(d.entropy() for d in cam_dists) + sum(d.entropy() for d in guard_dists)
        return cam_actions, guard_actions, log_prob, entropy, value, hx, cx


# ── Instantiate ──────────────────────────────────────────────────────────
G = CFG.grid_rows * CFG.grid_cols

architect_net = ArchitectNet(
    in_ch=CFG.input_channels, lstm_hidden=CFG.lstm_hidden,
    n_cameras=CFG.n_cameras, n_guards=CFG.n_guards, grid_size=G
).to(DEV_ARCH)

robber_net = RobberNet(
    in_ch=CFG.input_channels, lstm_hidden=CFG.lstm_hidden
).to(DEV_ROBBER)

arch_params   = sum(p.numel() for p in architect_net.parameters())
robber_params = sum(p.numel() for p in robber_net.parameters())
print(f'✅ Architect: {arch_params/1e6:.2f}M params on {DEV_ARCH}')
print(f'✅ Robber:    {robber_params/1e6:.2f}M params on {DEV_ROBBER}')

✅ Architect: 8.52M params on cuda:0
✅ Robber:    7.02M params on cuda:1


## 📊 7. ELO Rating System

In [44]:
class EloRater:
    """Track agent strength over time using the ELO system."""
    def __init__(self, k=32.0, initial=1200.0):
        self.k = k
        self.arch_elo   = initial
        self.robber_elo = initial
        self.history    = []  # (episode, arch_elo, robber_elo)

    def expected(self, rating_a, rating_b):
        return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))

    def update(self, robber_won: bool, episode: int):
        """Update ELO after one episode outcome."""
        exp_arch   = self.expected(self.arch_elo, self.robber_elo)
        exp_robber = 1.0 - exp_arch

        score_robber = 1.0 if robber_won else 0.0
        score_arch   = 1.0 - score_robber

        self.arch_elo   += self.k * (score_arch   - exp_arch)
        self.robber_elo += self.k * (score_robber - exp_robber)
        self.history.append((episode, self.arch_elo, self.robber_elo))

    @property
    def diff(self):
        return self.robber_elo - self.arch_elo  # positive = robber dominating

elo = EloRater()
print('✅ ELO tracker ready')

✅ ELO tracker ready


## 🎓 8. Curriculum Manager

In [45]:
class CurriculumManager:
    """
    Tracks robber win rate over recent episodes.
    When robber wins too easily → increase difficulty (more cameras/guards).
    When architect wins too easily → hold stage until robber catches up.
    This creates a balanced arms race so BOTH agents keep improving.
    """
    def __init__(self, stages: list, win_thresh=0.55, window=200):
        self.stages = stages
        self.stage_idx = 0
        self.win_thresh = win_thresh
        self.window = window
        self.recent_wins = deque(maxlen=window)
        self.advance_log = []

    @property
    def current_stage(self):
        return self.stages[self.stage_idx]

    @property
    def win_rate(self):
        if not self.recent_wins: return 0.5
        return sum(self.recent_wins) / len(self.recent_wins)

    def record(self, robber_won: bool, episode: int):
        self.recent_wins.append(int(robber_won))
        # Advance if robber is winning too easily
        if (self.win_rate > self.win_thresh
                and len(self.recent_wins) >= self.window // 2
                and self.stage_idx < len(self.stages) - 1):
            self.stage_idx += 1
            self.advance_log.append((episode, self.current_stage['name']))
            self.recent_wins.clear()
            print(f'  📈 Curriculum advance → Stage {self.stage_idx}: {self.current_stage["name"]}')
            return True
        return False

    def env_kwargs(self):
        return {
            'n_cameras': self.current_stage['n_cameras'],
            'n_guards':  self.current_stage['n_guards'],
        }

curriculum = CurriculumManager(CFG.curriculum_stages, CFG.curriculum_win_thresh)
print(f'✅ Curriculum ready. Starting stage: {curriculum.current_stage["name"]}')

✅ Curriculum ready. Starting stage: Rookie


## ⚔️ 9. Self-Play Pool

In [46]:
class SelfPlayPool:
    """
    Maintains a pool of past agent snapshots.
    When training, the current agent sometimes plays against a past version
    instead of always the current opponent. This prevents strategy collapse
    (where agents find one Nash equilibrium and stop improving).
    """
    def __init__(self, max_size: int, use_latest_prob: float, device):
        self.pool = deque(maxlen=max_size)
        self.use_latest_prob = use_latest_prob
        self.device = device

    def add(self, model: nn.Module):
        snapshot = copy.deepcopy(model).cpu()
        snapshot.eval()
        self.pool.append(snapshot)

    def sample(self, current_model: nn.Module) -> nn.Module:
        """Return the model to use as opponent."""
        if not self.pool or random.random() < self.use_latest_prob:
            return current_model  # play vs self
        snap = random.choice(list(self.pool))
        return snap.to(self.device)

arch_pool   = SelfPlayPool(CFG.selfplay_pool_size, CFG.selfplay_use_latest, DEV_ARCH)
robber_pool = SelfPlayPool(CFG.selfplay_pool_size, CFG.selfplay_use_latest, DEV_ROBBER)
print('✅ Self-play pools ready')

✅ Self-play pools ready


## 🔄 10. PPO Trainer

**This is what v1 was completely missing.** The original notebook had no actual PPO loop — just forward passes with no gradient updates based on rollout returns.

In [47]:
from collections import defaultdict

class RolloutBuffer:
    """Stores one rollout of (obs, action, reward, ...) for PPO updates."""
    def __init__(self):
        self.obs, self.actions, self.rewards = [], [], []
        self.log_probs, self.values, self.dones = [], [], []

    def add(self, obs, action, reward, log_prob, value, done):
        self.obs.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.dones.append(done)

    def clear(self):
        self.__init__()

    def compute_returns(self, last_value, gamma, gae_lambda):
        """Compute GAE advantages and discounted returns."""
        T = len(self.rewards)
        advantages = np.zeros(T, dtype=np.float32)
        last_adv = 0.0
        for t in reversed(range(T)):
            nv = last_value if t == T-1 else self.values[t+1]
            delta = self.rewards[t] + gamma * nv * (1 - self.dones[t]) - self.values[t]
            last_adv = delta + gamma * gae_lambda * (1 - self.dones[t]) * last_adv
            advantages[t] = last_adv
        returns = advantages + np.array(self.values, dtype=np.float32)
        return advantages, returns


class PPOAgent:
    def __init__(self, net, optimizer, device, cfg: Config, agent_name='agent'):
        self.net = net
        self.opt = optimizer
        self.device = device
        self.cfg = cfg
        self.name = agent_name
        self.buffer = RolloutBuffer()
        self.hx, self.cx = net.initial_state(batch=1, device=device)
        self.train_stats = defaultdict(list)

    def reset_hidden(self):
        self.hx, self.cx = self.net.initial_state(batch=1, device=self.device)

    def detach_hidden(self):
        """Detach LSTM state between rollouts (keep values, drop gradients)."""
        self.hx = self.hx.detach()
        self.cx = self.cx.detach()

    def _ppo_update(self, obs_t, actions_t, old_logp_t, advantages_t, returns_t):
        """Single PPO minibatch update. Returns loss dict."""
        # Recompute logprobs & values from scratch on this minibatch
        # (We use a simplified forward without LSTM for the update phase
        # to avoid sequence-dependency complexity in minibatching)
        dummy_hx = torch.zeros(obs_t.shape[0], self.cfg.lstm_hidden, device=self.device)
        dummy_cx = torch.zeros_like(dummy_hx)

        if self.name == 'robber':
            logits, values, _, _ = self.net(obs_t, dummy_hx, dummy_cx)
            dist = Categorical(logits=logits)
            new_logp = dist.log_prob(actions_t)
            entropy  = dist.entropy().mean()
        else:  # architect
            cam_logits, guard_logits, values, _, _ = self.net(obs_t, dummy_hx, dummy_cx)
            # actions_t shape: (B, n_cameras + n_guards)
            nc = self.cfg.n_cameras
            ng = self.cfg.n_guards
            new_logp = sum(
                Categorical(logits=cam_logits[:,i,:]).log_prob(actions_t[:,i])
                for i in range(nc)
            )
            new_logp += sum(
                Categorical(logits=guard_logits[:,j,:]).log_prob(actions_t[:,nc+j])
                for j in range(ng)
            )
            entropy = (sum(Categorical(logits=cam_logits[:,i,:]).entropy() for i in range(nc)) +
                       sum(Categorical(logits=guard_logits[:,j,:]).entropy() for j in range(ng))).mean()

        # PPO clip loss
        ratio = torch.exp(new_logp - old_logp_t)
        pg1   = ratio * advantages_t
        pg2   = torch.clamp(ratio, 1-self.cfg.clip_eps, 1+self.cfg.clip_eps) * advantages_t
        pg_loss  = -torch.min(pg1, pg2).mean()
        val_loss = F.mse_loss(values, returns_t)
        ent_loss = -self.cfg.entropy_coef * entropy
        loss = pg_loss + self.cfg.value_coef * val_loss + ent_loss

        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.net.parameters(), self.cfg.max_grad_norm)
        self.opt.step()

        return {
            'pg_loss': pg_loss.item(),
            'val_loss': val_loss.item(),
            'entropy': entropy.item(),
            'total_loss': loss.item()
        }

    def update_from_buffer(self, last_value):
        """Run PPO epochs over the collected rollout."""
        adv, ret = self.buffer.compute_returns(last_value, self.cfg.gamma, self.cfg.gae_lambda)

        # Normalize advantages
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        obs_arr     = np.array(self.buffer.obs, dtype=np.float32)
        actions_arr = np.array(self.buffer.actions)
        logp_arr    = np.array(self.buffer.log_probs, dtype=np.float32)

        obs_t     = torch.tensor(obs_arr, device=self.device)
        actions_t = torch.tensor(actions_arr, dtype=torch.long, device=self.device)
        logp_t    = torch.tensor(logp_arr, device=self.device)
        adv_t     = torch.tensor(adv, device=self.device)
        ret_t     = torch.tensor(ret, device=self.device)

        T = obs_t.shape[0]
        epoch_stats = defaultdict(list)
        for _ in range(self.cfg.ppo_epochs):
            idx = torch.randperm(T)
            for start in range(0, T, self.cfg.minibatch_sz):
                mb = idx[start:start+self.cfg.minibatch_sz]
                stats = self._ppo_update(
                    obs_t[mb], actions_t[mb], logp_t[mb], adv_t[mb], ret_t[mb]
                )
                for k,v in stats.items():
                    epoch_stats[k].append(v)

        for k,v in epoch_stats.items():
            self.train_stats[k].append(np.mean(v))

        self.buffer.clear()
        self.detach_hidden()


# Create agents
arch_agent = PPOAgent(
    architect_net,
    optim.Adam(architect_net.parameters(), lr=CFG.lr_arch, eps=1e-5),
    DEV_ARCH, CFG, agent_name='architect'
)
robber_agent = PPOAgent(
    robber_net,
    optim.Adam(robber_net.parameters(), lr=CFG.lr_robber, eps=1e-5),
    DEV_ROBBER, CFG, agent_name='robber'
)

print('✅ PPO agents ready')

✅ PPO agents ready


## 🏋️ 11. Main Training Loop

In [48]:
def obs_to_tensor(obs, device):
    """Convert environment observation to a batched tensor ready for the network."""
    obs_np = process_observation(obs)
    return torch.tensor(obs_np, dtype=torch.float32, device=device).unsqueeze(0)


def run_episode(arch_agent, robber_agent, env, train=True):
    """
    Run one full episode. Both agents act, collect rollout.
    Returns episode stats dict.
    """
    raw_obs = env.reset(**curriculum.env_kwargs()) if not USING_ORIGINAL_ENV else env.reset()
    arch_agent.reset_hidden()
    robber_agent.reset_hidden()

    ep_reward_robber = 0.0
    ep_reward_arch   = 0.0
    step_count = 0
    info_log = {'caught': 0, 'robbed': 0, 'alarms': 0}

    obs = process_observation(raw_obs)

    while True:
        obs_a = obs_to_tensor(obs, DEV_ARCH)
        obs_r = obs_to_tensor(obs, DEV_ROBBER)

        # Architect acts first (sets up defenses)
        cam_acts, guard_acts, arch_logp, arch_ent, arch_val, arch_hx, arch_cx = \
            arch_agent.net.act(obs_a, arch_agent.hx, arch_agent.cx)
        arch_agent.hx, arch_agent.cx = arch_hx, arch_cx

        # Apply architect actions to environment
        if not USING_ORIGINAL_ENV:
            # Standalone env: camera placement only on first step
            if step_count == 0:
                env.step_architect(cam_acts, guard_acts)
            else:
                env.step_architect([], guard_acts)
        else:
            # Original environment: adapt method names based on available API
            if step_count == 0:
                # Place cameras (try common method names)
                if hasattr(env, 'place_cameras'):
                    env.place_cameras(cam_acts)
                elif hasattr(env, 'set_camera_positions'):
                    env.set_camera_positions(cam_acts)
                # Move guards (initial positions)
                if hasattr(env, 'move_guards'):
                    env.move_guards(guard_acts)
                elif hasattr(env, 'set_guard_positions'):
                    env.set_guard_positions(guard_acts)
            else:
                # Move guards only
                if hasattr(env, 'move_guards'):
                    env.move_guards(guard_acts)

        # Robber acts
        rob_act, rob_logp, rob_val, rob_hx, rob_cx = \
            robber_agent.net.act(obs_r, robber_agent.hx, robber_agent.cx)
        robber_agent.hx, robber_agent.cx = rob_hx, rob_cx

        # Take robber step in environment
        if not USING_ORIGINAL_ENV:
            raw_obs_next, robber_rew, done, info = env.step_robber(rob_act)
        else:
            # Original environment: try different step methods
            if hasattr(env, 'step_robber'):
                raw_obs_next, robber_rew, done, info = env.step_robber(rob_act)
            elif hasattr(env, 'step'):
                # Single step method returns (obs, reward, done, info)
                raw_obs_next, robber_rew, done, info = env.step(rob_act)
            else:
                raise AttributeError("Environment has no step_robber or step method")

        obs_next = process_observation(raw_obs_next)

        # --- Reward scaling to help robber learn ---
        robber_rew = robber_rew * 0.5
        arch_rew = -robber_rew

        ep_reward_robber += robber_rew
        ep_reward_arch   += arch_rew
        info_log['caught'] += int(info.get('caught', False))
        info_log['robbed'] += int(info.get('robbed', False))
        info_log['alarms'] += int(info.get('alarm', False))

        if train:
            arch_action_vec = np.array(cam_acts + guard_acts, dtype=np.int64)
            robber_agent.buffer.add(obs, rob_act, robber_rew,
                                    rob_logp.item() if hasattr(rob_logp,'item') else float(rob_logp),
                                    rob_val.item() if hasattr(rob_val,'item') else float(rob_val),
                                    done)
            arch_agent.buffer.add(obs, arch_action_vec, arch_rew,
                                   arch_logp.item() if hasattr(arch_logp,'item') else float(arch_logp),
                                   arch_val.item() if hasattr(arch_val,'item') else float(arch_val),
                                   done)

        obs = obs_next
        step_count += 1

        if done:
            break

    robber_won = (info_log['caught'] == 0 and info_log['robbed'] > 0) or ep_reward_robber > 0

    return {
        'robber_reward': ep_reward_robber,
        'arch_reward':   ep_reward_arch,
        'robber_won':    robber_won,
        'steps':         step_count,
        '_last_obs':     obs,
        **info_log
    }

print('✅ Episode runner ready')

✅ Episode runner ready


In [49]:
print(dir(env))

['ACTIONS', 'ACTION_NAMES', 'NUM_SOLVER_ACTIONS', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_get_observation', '_initial_dist', '_is_valid_placement', '_prev_dist', '_reset_layout', 'budget', 'cameras', 'config', 'detection_events', 'distance_map', 'done', 'get_architect_reward', 'get_environment_state', 'get_state_tensor', 'grid', 'guards', 'is_level_valid', 'render_text', 'reset', 'set_layout', 'solver_detected', 'solver_path', 'solver_pos', 'step', 'tick', 'vault_reached', 'visibility_map', 'walls']


In [ ]:
# ── Training orchestration ─────────────────────────────────────────────────
episode_metrics = defaultdict(list)
best_robber_win_rate = 0.0
recent_outcomes = deque(maxlen=100)
start_time = time.time()

env = make_env(CFG)

print('=' * 60)
print(f'🚀 Starting training for {CFG.total_episodes} episodes')
print(f'   Checkpoint every {CFG.checkpoint_every} episodes → HuggingFace')
print(f'   Curriculum: {len(CFG.curriculum_stages)} stages')
print('=' * 60)

import os
from huggingface_hub import hf_hub_download

START_EPISODE = 1
if HF_AVAILABLE:
    print('Attempting to resume from latest checkpoint on HF...')
    try:
        refs = api.list_repo_files(repo_id=REPO_ID)
        ep_folders = [f.split('/')[0] for f in refs if f.startswith('ep') and 'metrics.json' in f]
        if ep_folders:
            ep_folders.sort()
            latest_tag = ep_folders[-1]
            print(f'➡️ Found latest checkpoint: {latest_tag}')
            m_path = hf_hub_download(repo_id=REPO_ID, filename=f'{latest_tag}/metrics.json')
            a_path = hf_hub_download(repo_id=REPO_ID, filename=f'{latest_tag}/architect.pt')
            r_path = hf_hub_download(repo_id=REPO_ID, filename=f'{latest_tag}/robber.pt')
            with open(m_path, 'r') as f:
                m_data = json.load(f)
            START_EPISODE = m_data['episode'] + 1
            elo.arch_elo = m_data.get('arch_elo', elo.arch_elo)
            elo.robber_elo = m_data.get('robber_elo', elo.robber_elo)
            c_stage = m_data.get('curriculum_stage')
            if c_stage:
                for idx, s in enumerate(curriculum.stages):
                    if s['name'] == c_stage:
                        curriculum.stage_idx = idx
                        break
            def strip_prefix(state_dict):
                return {k.replace('module.', ''): v for k, v in state_dict.items()}
            architect_net.load_state_dict(strip_prefix(torch.load(a_path)), strict=False)
            robber_net.load_state_dict(strip_prefix(torch.load(r_path)), strict=False)
            print(f'✅ Successfully loaded state. Resuming from episode {START_EPISODE}')
        else:
            print('No checkpoints found, starting from scratch.')
    except Exception as e:
        print(f'⚠️ Could not resume: {e}')

for ep in range(START_EPISODE, CFG.total_episodes + 1):
    # ── Run episode ────────────────────────────────────────────
    stats = run_episode(arch_agent, robber_agent, env, train=True)
    recent_outcomes.append(stats['robber_won'])

    # ── Update ELO ────────────────────────────────────────────
    elo.update(stats['robber_won'], ep)

    # ── Curriculum check ─────────────────────────────────────
    curriculum.record(stats['robber_won'], ep)

    # ── Log metrics ───────────────────────────────────────────
    for k, v in stats.items():
        episode_metrics[k].append(v)

    # ── PPO update every rollout_len steps ─────────────────────
    if len(robber_agent.buffer.rewards) >= CFG.rollout_len:
        # Bootstrap value from CURRENT obs (don't reset env mid-episode!)
        with torch.no_grad():
            last_obs = stats['_last_obs']
            obs_r_t = obs_to_tensor(last_obs, DEV_ROBBER)
            obs_a_t = obs_to_tensor(last_obs, DEV_ARCH)
            bstrap_hx_r, bstrap_cx_r = robber_net.initial_state(device=DEV_ROBBER)
            bstrap_hx_a, bstrap_cx_a = architect_net.initial_state(device=DEV_ARCH)
            _, last_val_r, _, _ = robber_net(obs_r_t, bstrap_hx_r, bstrap_cx_r)
            _, _, last_val_a, _, _ = architect_net(obs_a_t, bstrap_hx_a, bstrap_cx_a)

        robber_agent.update_from_buffer(last_val_r.item())
        arch_agent.update_from_buffer(last_val_a.item())

    # ── Self-play pool snapshots ──────────────────────────────
    if ep % 200 == 0:
        arch_pool.add(architect_net)
        robber_pool.add(robber_net)

    # ── Console log ───────────────────────────────────────────
    if ep % CFG.log_every == 0:
        win_rate = sum(recent_outcomes) / len(recent_outcomes)
        rob_r    = np.mean(episode_metrics['robber_reward'][-CFG.log_every:])
        arch_r   = np.mean(episode_metrics['arch_reward'][-CFG.log_every:])
        elapsed  = time.time() - start_time
        stage    = curriculum.current_stage['name']
        print(f'Ep {ep:5d} | stage={stage:12s} | '
              f'robber_win={win_rate:.2f} | '
              f'rob_r={rob_r:+.1f} arch_r={arch_r:+.1f} | '
              f'ELO diff={elo.diff:+.0f} | '
              f'elapsed={elapsed/60:.1f}m')

    # ── Checkpoint ────────────────────────────────────────────
    if ep % CFG.checkpoint_every == 0:
        win_rate = sum(recent_outcomes) / max(len(recent_outcomes), 1)
        metrics_snapshot = {
            'robber_win_rate': win_rate,
            'elo_diff':        elo.diff,
            'arch_elo':        elo.arch_elo,
            'robber_elo':      elo.robber_elo,
            'curriculum_stage':curriculum.current_stage['name'],
            'mean_robber_reward': float(np.mean(episode_metrics['robber_reward'][-100:])),
        }
        save_to_hf(architect_net, robber_net, metrics_snapshot, ep)

print('\n✅ Training complete!')

🚀 Starting training for 20000 episodes
   Checkpoint every 500 episodes → HuggingFace
   Curriculum: 4 stages
Attempting to resume from latest checkpoint on HF...
No checkpoints found, starting from scratch.
Ep    50 | stage=Rookie       | robber_win=0.96 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+477 | elapsed=2.6m
  📈 Curriculum advance → Stage 1: Intermediate
Ep   100 | stage=Intermediate | robber_win=0.98 | rob_r=+0.4 arch_r=-0.4 | ELO diff=+609 | elapsed=5.3m
Ep   150 | stage=Intermediate | robber_win=1.00 | rob_r=+0.4 arch_r=-0.4 | ELO diff=+685 | elapsed=7.9m
  📈 Curriculum advance → Stage 2: Expert
Ep   200 | stage=Expert       | robber_win=1.00 | rob_r=+0.4 arch_r=-0.4 | ELO diff=+737 | elapsed=10.6m
Ep   250 | stage=Expert       | robber_win=1.00 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+778 | elapsed=13.2m
  📈 Curriculum advance → Stage 3: Master
Ep   300 | stage=Master       | robber_win=1.00 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+810 | elapsed=15.9m
Ep   350 | stage=Master       | robb

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ HF checkpoint pushed: ep00500
Ep   550 | stage=Master       | robber_win=0.92 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+457 | elapsed=29.1m
Ep   600 | stage=Master       | robber_win=0.90 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+413 | elapsed=31.8m
Ep   650 | stage=Master       | robber_win=0.89 | rob_r=+0.2 arch_r=-0.2 | ELO diff=+401 | elapsed=34.3m
Ep   700 | stage=Master       | robber_win=0.83 | rob_r=+0.1 arch_r=-0.1 | ELO diff=+300 | elapsed=37.0m
Ep   750 | stage=Master       | robber_win=0.83 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+292 | elapsed=39.6m
Ep   800 | stage=Master       | robber_win=0.90 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+416 | elapsed=42.2m
Ep   850 | stage=Master       | robber_win=0.91 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+347 | elapsed=44.8m
Ep   900 | stage=Master       | robber_win=0.90 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+382 | elapsed=47.4m
Ep   950 | stage=Master       | robber_win=0.92 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+478 | elapsed=50.0m
Ep  1000 | stage=Mast

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  ✅ HF checkpoint pushed: ep03000
Ep  3050 | stage=Master       | robber_win=0.78 | rob_r=+0.4 arch_r=-0.4 | ELO diff=+522 | elapsed=160.1m
Ep  3100 | stage=Master       | robber_win=1.00 | rob_r=+0.4 arch_r=-0.4 | ELO diff=+632 | elapsed=162.7m
Ep  3150 | stage=Master       | robber_win=1.00 | rob_r=+0.4 arch_r=-0.4 | ELO diff=+700 | elapsed=165.4m
Ep  3250 | stage=Master       | robber_win=0.97 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+643 | elapsed=170.6m
Ep  3300 | stage=Master       | robber_win=0.95 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+511 | elapsed=173.2m
Ep  3350 | stage=Master       | robber_win=0.94 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+527 | elapsed=175.8m
Ep  3400 | stage=Master       | robber_win=0.97 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+598 | elapsed=178.4m
Ep  3450 | stage=Master       | robber_win=0.99 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+677 | elapsed=181.0m
Ep  3500 | stage=Master       | robber_win=1.00 | rob_r=+0.3 arch_r=-0.3 | ELO diff=+731 | elapsed=183.6m


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ HF checkpoint pushed: ep03500


## 📈 12. Training Dashboard

In [ ]:
def plot_training_dashboard(episode_metrics, elo_history, curriculum_advances):
    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor('#0d0d1a')
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    style = {'color': '#00ff9f', 'linewidth': 1.5}
    ax_style = {'facecolor': '#1a1a2e', 'labelcolor': '#aaaacc',
                'titlecolor': '#ffffff', 'tickcolor': '#555577'}

    def style_ax(ax, title):
        ax.set_facecolor(ax_style['facecolor'])
        ax.set_title(title, color=ax_style['titlecolor'], fontsize=11, pad=8)
        ax.tick_params(colors=ax_style['tickcolor'])
        ax.yaxis.label.set_color(ax_style['labelcolor'])
        ax.xaxis.label.set_color(ax_style['labelcolor'])
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')

    eps = np.arange(1, len(episode_metrics['robber_reward'])+1)

    def smooth(x, w=50):
        if len(x) < w: return x
        return np.convolve(x, np.ones(w)/w, mode='valid')

    # 1. Robber vs Architect rewards
    ax1 = fig.add_subplot(gs[0, :])
    style_ax(ax1, '📊 Reward Over Training')
    ax1.plot(smooth(episode_metrics['robber_reward']), color='#00ff9f', lw=1.5, label='Robber')
    ax1.plot(smooth(episode_metrics['arch_reward']),   color='#ff4466', lw=1.5, label='Architect')
    ax1.axhline(0, color='#ffffff33', lw=0.5)
    for ep_adv, sname in curriculum_advances:
        ax1.axvline(ep_adv, color='#ffaa00', lw=1, linestyle='--', alpha=0.6)
    ax1.legend(facecolor='#1a1a2e', edgecolor='#555577', labelcolor='white')
    ax1.set_xlabel('Episode')

    # 2. ELO ratings
    ax2 = fig.add_subplot(gs[1, 0])
    style_ax(ax2, '⚡ ELO Ratings')
    if elo_history:
        eps_elo = [h[0] for h in elo_history]
        ax2.plot(eps_elo, [h[1] for h in elo_history], color='#ff4466', lw=1.5, label='Architect')
        ax2.plot(eps_elo, [h[2] for h in elo_history], color='#00ff9f', lw=1.5, label='Robber')
    ax2.legend(facecolor='#1a1a2e', edgecolor='#555577', labelcolor='white')

    # 3. Win rate
    ax3 = fig.add_subplot(gs[1, 1])
    style_ax(ax3, '🏆 Robber Win Rate (100-ep rolling)')
    wins = episode_metrics['robber_won']
    rolling = [np.mean(wins[max(0,i-100):i+1]) for i in range(len(wins))]
    ax3.plot(rolling, color='#00ccff', lw=1.5)
    ax3.axhline(0.5, color='#ffffff44', lw=1, linestyle='--')
    ax3.set_ylim(0, 1)

    # 4. Alarms triggered
    ax4 = fig.add_subplot(gs[1, 2])
    style_ax(ax4, '🚨 Alarms Triggered per Episode')
    ax4.plot(smooth(episode_metrics['alarms']), color='#ffaa00', lw=1.5)

    # 5. Vaults robbed per episode
    ax5 = fig.add_subplot(gs[2, 0])
    style_ax(ax5, '💰 Vaults Robbed per Episode')
    ax5.plot(smooth(episode_metrics['robbed']), color='#aa88ff', lw=1.5)

    # 6. Episode length
    ax6 = fig.add_subplot(gs[2, 1])
    style_ax(ax6, '⏱️ Episode Length')
    ax6.plot(smooth(episode_metrics['steps']), color='#44ddff', lw=1.5)

    # 7. Curriculum progress
    ax7 = fig.add_subplot(gs[2, 2])
    style_ax(ax7, '🎓 Curriculum Stage')
    stage_names = [s['name'] for s in CFG.curriculum_stages]
    ax7.set_yticks(range(len(stage_names)))
    ax7.set_yticklabels(stage_names)
    if curriculum.advance_log:
        stage_trace_x, stage_trace_y = [], []
        cur = 0
        prev_ep = 0
        for ep_adv, sname in curriculum.advance_log:
            stage_trace_x += [prev_ep, ep_adv]
            stage_trace_y += [cur, cur]
            cur = stage_names.index(sname)
            prev_ep = ep_adv
        stage_trace_x.append(len(wins))
        stage_trace_y.append(cur)
        ax7.step(stage_trace_x, stage_trace_y, color='#ffcc44', lw=2, where='post')

    fig.suptitle('🏛️ HEIST ARCHITECT v2 — Training Dashboard',
                 color='white', fontsize=16, fontweight='bold', y=0.98)
    plt.savefig('/kaggle/working/training_dashboard.png', dpi=150, bbox_inches='tight',
                facecolor='#0d0d1a')
    plt.show()
    print('✅ Dashboard saved to /kaggle/working/training_dashboard.png')


plot_training_dashboard(episode_metrics, elo.history, curriculum.advance_log)

## 🎬 13. Visualize a Head-to-Head Episode

In [15]:
def visualize_episode(arch_net, robber_net, env, max_steps=100):
    """Render frames of one greedy episode as a matplotlib animation."""
    arch_net.eval()
    robber_net.eval()

    obs = env.reset(**curriculum.env_kwargs()) if not USING_ORIGINAL_ENV else env.reset()
    hx_a, cx_a = arch_net.initial_state(device=DEV_ARCH)
    hx_r, cx_r = robber_net.initial_state(device=DEV_ROBBER)

    frames = []

    for _ in range(max_steps):
        obs_a = obs_to_tensor(obs, DEV_ARCH)
        obs_r = obs_to_tensor(obs, DEV_ROBBER)

        cam_acts, guard_acts, _, _, _, hx_a, cx_a = arch_net.act(obs_a, hx_a, cx_a, greedy=True)
        if not USING_ORIGINAL_ENV:
            env.step_architect(cam_acts, guard_acts)

        rob_act, _, _, hx_r, cx_r = robber_net.act(obs_r, hx_r, cx_r, greedy=True)
        obs, _, done, _ = env.step_robber(rob_act)

        # Capture grid state
        if not USING_ORIGINAL_ENV:
            frames.append(env.grid.copy())
        if done:
            break

    if not frames:
        print('No frames to visualize (original env not returning grid).')
        return

    # Plot a sample of frames
    n_show = min(12, len(frames))
    step = max(1, len(frames) // n_show)
    selected = frames[::step][:n_show]

    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(
        ['#0d0d1a', '#334455', '#ffd700', '#00aaff', '#ff4444', '#00ff9f', '#ff8800']
    )

    fig, axes = plt.subplots(3, 4, figsize=(16, 12), facecolor='#0d0d1a')
    fig.suptitle('🎬 Episode Replay — Greedy Policy', color='white', fontsize=14)

    for i, (ax, frame) in enumerate(zip(axes.flatten(), selected)):
        ax.imshow(frame, cmap=cmap, vmin=0, vmax=6, interpolation='nearest')
        ax.set_title(f'Step {i*step}', color='#aaaacc', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('/kaggle/working/episode_replay.png', dpi=120, bbox_inches='tight',
                facecolor='#0d0d1a')
    plt.show()
    print('✅ Episode replay saved')

    # Push viz to HF
    if not HF_AVAILABLE:
        print('⚠️  HF offline — visuals saved locally only')
        return
    try:
        api.upload_file(
            path_or_fileobj='/kaggle/working/episode_replay.png',
            path_in_repo='visuals/episode_replay_final.png',
            repo_id=REPO_ID, token=HF_TOKEN
        )
        api.upload_file(
            path_or_fileobj='/kaggle/working/training_dashboard.png',
            path_in_repo='visuals/training_dashboard.png',
            repo_id=REPO_ID, token=HF_TOKEN
        )
        print('✅ Visuals pushed to HuggingFace')
    except Exception as e:
        print(f'⚠️  Could not push visuals: {e}')


visualize_episode(architect_net, robber_net, env)

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

## 💾 14. Final Save

In [ ]:
# Final checkpoint push
final_win_rate = sum(episode_metrics['robber_won'][-200:]) / max(len(episode_metrics['robber_won'][-200:]), 1)
save_to_hf(architect_net, robber_net, {
    'robber_win_rate': final_win_rate,
    'elo_diff': elo.diff,
    'arch_elo': elo.arch_elo,
    'robber_elo': elo.robber_elo,
    'final_curriculum_stage': curriculum.current_stage['name'],
    'total_episodes': CFG.total_episodes,
}, episode=CFG.total_episodes)

print('\n🏁 Done! Summary:')
print(f'  Final robber win rate: {final_win_rate:.2%}')
print(f'  Architect ELO: {elo.arch_elo:.0f}  |  Robber ELO: {elo.robber_elo:.0f}')
print(f'  Curriculum reached: {curriculum.current_stage["name"]}')
print(f'  Models saved at: https://huggingface.co/{REPO_ID}')